In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import deque
import time

In [7]:
# 1. Load your best model
model = YOLO('../best_yolov8_coral_reef/runs/detect/reef_coral/weights/best.pt')

# 2. Setup Video
video_path = '../yolov8_model/videos/Match 12 (R4) - 2025 Iowa Regional.mp4'
cap = cv2.VideoCapture(video_path)

In [8]:

# ---------------- CONFIG ----------------
OUTPUT_PATH = 'debug_state_based.mp4'

SLOT_CAPACITIES = [3, 1, 2, 2, 1, 3]
NUM_SLOTS = len(SLOT_CAPACITIES)
NUM_REEFS = 2  # adjust if needed

# Detection / smoothing parameters
HISTORY_LEN = 5          # frames for smoothing
MIN_CONF = 0.5            # confidence threshold
TOP_ZONE_RATIO = 0.20     # top portion of reef for scoring
FRAME_PERSISTENCE = 3     # coral must be present for >= this many frames

# ----------------------------------------

# Load video & YOLO model

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cutoff_frame = total_frames - int(49 * fps)

crop_h = int(height * (2/5))
start_y = height - crop_h

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, crop_h))

# ---------------- STATE ----------------
reef_grids = [[0]*NUM_SLOTS for _ in range(NUM_REEFS)]
history = [[deque(maxlen=HISTORY_LEN) for _ in range(NUM_SLOTS)] for _ in range(NUM_REEFS)]
max_seen = [[0]*NUM_SLOTS for _ in range(NUM_REEFS)]
total_points = 0
current_frame_idx = 0
scoring_log = []

# ---------------- HELPERS ----------------
def get_center(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2, (y1 + y2) / 2)

def assign_to_reef_and_slot(cx, cy, reefs):
    for r_idx, r in enumerate(reefs):
        rx1, ry1, rx2, ry2 = r
        rw, rh = rx2 - rx1, ry2 - ry1

        # Top portion of reef
        if (rx1 < cx < rx2) and (ry1 < cy < ry1 + rh * TOP_ZONE_RATIO):
            rel_x = cx - rx1
            slot_width = rw / NUM_SLOTS
            slot_margin = slot_width * 0.1
            slot = int((rel_x - slot_margin) / slot_width)
            slot = max(0, min(slot, NUM_SLOTS - 1))
            return r_idx, slot
    return None, None

# ---------------- MAIN LOOP ----------------
while cap.isOpened():
    ret, frame = cap.read()
    if not ret or current_frame_idx >= cutoff_frame:
        break

    cropped = frame[start_y:height, 0:width]

    results = model(cropped, conf=MIN_CONF, verbose=False)

    annotated_frame = results[0].plot()

    # --- DETECTIONS ---
    boxes = results[0].boxes.xyxy.cpu().numpy() if results[0].boxes else []
    clss  = results[0].boxes.cls.cpu().numpy().astype(int) if results[0].boxes else []
    confs = results[0].boxes.conf.cpu().numpy() if results[0].boxes else []

    reefs = [boxes[i] for i, c in enumerate(clss) if model.names[c] == 'reef']
    reefs.sort(key=lambda x: x[0])

    # --- COUNT CORAL PER SLOT ---
    raw_counts = [[0]*NUM_SLOTS for _ in range(NUM_REEFS)]

    for i, c in enumerate(clss):
        if model.names[c] != 'coral' or confs[i] < MIN_CONF:
            continue
        cx, cy = get_center(boxes[i])
        r_idx, slot = assign_to_reef_and_slot(cx, cy, reefs)
        if r_idx is not None and r_idx < NUM_REEFS:
            raw_counts[r_idx][slot] += 1

    # --- SMOOTH COUNTS ---
    smoothed_counts = [[0]*NUM_SLOTS for _ in range(NUM_REEFS)]
    for r in range(NUM_REEFS):
        for s in range(NUM_SLOTS):
            history[r][s].append(raw_counts[r][s])
            smoothed_counts[r][s] = int(round(np.median(history[r][s])))

    # --- SCORE EVENTS (capacity-aware) ---
    for r in range(NUM_REEFS):
        for s in range(NUM_SLOTS):
            curr = smoothed_counts[r][s]
            prev_max = max_seen[r][s]

            if curr > prev_max:
                diff = min(curr - prev_max, SLOT_CAPACITIES[s] - reef_grids[r][s])
                for _ in range(diff):
                    reef_grids[r][s] += 1
                    total_points += 4
                    seconds = current_frame_idx / fps
                    timestamp = f"{int(seconds//60):02d}:{int(seconds%60):02d}"
                    scoring_log.append((timestamp, r, s))
                    print(f"[{timestamp}] SCORE | Reef {r}, Slot {s} | Total: {total_points}")

                max_seen[r][s] = min(curr, SLOT_CAPACITIES[s])

    # --- DEBUG OVERLAY ---
    for r in range(NUM_REEFS):
        for s in range(NUM_SLOTS):
            text = f"{smoothed_counts[r][s]}/{SLOT_CAPACITIES[s]}"
            color = (0,255,0) if smoothed_counts[r][s] <= SLOT_CAPACITIES[s] else (0,0,255)
            cv2.putText(
                annotated_frame,
                text,
                (20 + s*60, 30 + r*40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                color,
                2
            )

    out.write(annotated_frame)
    current_frame_idx += 1

# ---------------- CLEANUP ----------------
cap.release()
out.release()

# ---------------- FINAL REPORT ----------------
print("\n" + "="*45)
print("SCORING TIMELINE")
print("-"*45)
for t, r, s in scoring_log:
    print(f"Time: {t} | Reef: {r} | Slot: {s}")

print("\nFINAL GRID:")
for r in range(NUM_REEFS):
    print(f"Reef {r}: {reef_grids[r]}")

[00:09] SCORE | Reef 1, Slot 0 | Total: 4
[00:15] SCORE | Reef 1, Slot 0 | Total: 8
[00:19] SCORE | Reef 1, Slot 4 | Total: 12
[00:26] SCORE | Reef 0, Slot 3 | Total: 16
[00:31] SCORE | Reef 1, Slot 2 | Total: 20
[00:31] SCORE | Reef 0, Slot 5 | Total: 24
[00:36] SCORE | Reef 1, Slot 2 | Total: 28
[00:40] SCORE | Reef 0, Slot 2 | Total: 32
[00:51] SCORE | Reef 0, Slot 3 | Total: 36
[01:02] SCORE | Reef 0, Slot 4 | Total: 40
[01:02] SCORE | Reef 1, Slot 5 | Total: 44
[01:12] SCORE | Reef 1, Slot 3 | Total: 48
[01:22] SCORE | Reef 1, Slot 1 | Total: 52
[01:30] SCORE | Reef 0, Slot 1 | Total: 56
[01:43] SCORE | Reef 0, Slot 0 | Total: 60
[01:49] SCORE | Reef 1, Slot 0 | Total: 64
[01:52] SCORE | Reef 1, Slot 3 | Total: 68
[01:53] SCORE | Reef 0, Slot 0 | Total: 72
[01:59] SCORE | Reef 0, Slot 2 | Total: 76
[02:04] SCORE | Reef 1, Slot 5 | Total: 80
[02:07] SCORE | Reef 1, Slot 5 | Total: 84
[02:09] SCORE | Reef 0, Slot 5 | Total: 88

SCORING TIMELINE
--------------------------------------

In [ ]:
print("\n" + "="*30)
print("FINAL REEF OCCUPANCY REPORT")
print("="*30)

for r_idx, slots in enumerate(reef_grids):
    # Only print reefs that actually had at least one coral detected
    if sum(slots) > 0:
        # Create a visual string: [0, 2, 1, 0, 0, 0] -> "Slot 0: 0 | Slot 1: 2 | Slot 2: 1 ..."
        status_str = " | ".join([f"Slot {i}: {val}" for i, val in enumerate(slots)])
        print(f"REEF {r_idx}: {status_str}")
        
        # Optional: Print total corals on this specific reef
        print(f"   -> Total Corals on Reef {r_idx}: {sum(slots)}")
        print("-" * 30)

print(f"GRAND TOTAL SCORE: {total_points} points")
print("="*30)

Starting analysis with custom slot caps: [3, 1, 2, 2, 1, 3]

SECTION    | SLOT 0 SLOT 1 SLOT 2 SLOT 3 SLOT 4 SLOT 5
CAPACITY   | (3)    (1)    (2)    (2)    (1)    (3)   
---------------------------------------------
GRAND TOTAL SCORE: 0 points
